### FAISS

Facebook Al Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning

In [2]:
#Importing the libraries
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
#Adding the file in a variable
documents = TextLoader('speech.txt').load()
print(type(documents))

#Checking the output type if it is a document or a dict etc
print(type(documents[0]))

<class 'list'>
<class 'langchain_core.documents.base.Document'>


In [9]:
# Initialize a text splitter in a variable and combine it with the documents
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
docs = text_splitter.split_documents(documents)
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [12]:
embeddings = OllamaEmbeddings(model='nomic-embed-text')
db = FAISS.from_documents(docs, embeddings)
db

In [17]:
query="What is the message about?"
docs=db.similarity_search(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

### As a retriever

We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [21]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

### Simmilarity Search with Score

There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Třerefore, a lower score is better.

In [22]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='6674586f-7a79-4e4f-97e6-505bf4466eed', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(1.1575155)),
 (Document(id='7e88da4f-df7e-4ca4-9470-9a2a2ac3a9ca', metadata={'source': 'spe

In [23]:
embedding_vector=embeddings.embed_query(query)
embedding_vector

[-0.029695999,
 0.042309515,
 -0.13959964,
 -0.042771332,
 0.08916384,
 0.030776026,
 0.03848977,
 -0.007887344,
 -0.008488316,
 -0.023084244,
 -0.013674774,
 0.07487072,
 0.08366295,
 0.044972688,
 -0.01545665,
 -0.02861567,
 -0.003560183,
 -0.055631306,
 -0.03512195,
 0.04687828,
 -0.021468667,
 -0.0073068296,
 -0.090895176,
 0.0041074622,
 0.09177176,
 0.047844876,
 0.03718729,
 0.019634927,
 -0.04220059,
 -0.0038993408,
 -0.0010977559,
 -0.01639049,
 -0.045986038,
 -0.04189021,
 -0.050123263,
 -0.024175929,
 0.015233624,
 0.038905587,
 -0.0016817204,
 -0.0017186956,
 0.012152745,
 0.014317046,
 -0.041792825,
 -0.052688338,
 0.043388925,
 0.005778754,
 -0.0032874628,
 0.011506124,
 -0.019688541,
 -0.036954053,
 0.04333415,
 -0.10096152,
 0.055343136,
 0.027106386,
 0.09066532,
 0.021705594,
 0.024402257,
 -0.010057051,
 -0.023774609,
 0.014988819,
 0.075794525,
 0.089142166,
 -0.025694637,
 0.08332981,
 0.06067492,
 -0.08197994,
 -0.017178815,
 -0.013447191,
 -0.0016370369,
 0.00044

In [24]:
docs_score = db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='6674586f-7a79-4e4f-97e6-505bf4466eed', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='7e88da4f-df7e-4ca4-9470-9a2a2ac3a9ca', metadata={'source': 'speech.txt'}, page_content='To 

In [25]:
#Saving and loading
db.save_local("faiss_index")

In [ ]:
new_df = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
docs = new_df.similarity_search(query)

In [28]:
docs

[Document(id='6674586f-7a79-4e4f-97e6-505bf4466eed', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='7e88da4f-df7e-4ca4-9470-9a2a2ac3a9ca', metadata={'source': 'speech.txt'}, page_content='To 